# 面试问题：Prefix Cache 怎样用 Radix Trie 复用 KV，同时防止跨租户和 ACL 版本泄漏？

        ## 可直接复述的回答主线

        1. Prefix Cache 的收益来自重复系统提示、工具说明和文档前缀，命中单位应是连续 Token 前缀。
2. 朴素全局缓存只按文本键查找，容易把不同租户或权限版本的 KV 当成可复用对象。
3. Radix Trie 按前缀逐段匹配，并把 tenant、role、ACL version 和模型版本放入命名空间。
4. 评估应输出逐请求命中长度、节省 Prefill Token、Trie 节点和拒绝复用原因。
5. 权限变化必须使旧缓存自然 miss 或显式失效，不能依赖调用方记得清理。
6. 生产实现还需要块引用计数、TTL、哈希碰撞防护、容量淘汰和调度器协同。

        后续实验会用同一批输入依次验证朴素方案、核心机制、失败边界和修正效果。

## 1. 真实案例与输入预览

案例包含财务与人力两个租户的六条 Agent 请求，字段包括 tenant、role、ACL version、系统提示、私有政策前缀和问题。相同文本故意出现在不同权限作用域，用于复现真实的跨租户/降权缓存风险；文本为脱敏教学样本。

In [1]:
from collections import defaultdict  # 按命名空间维护独立 Trie 和命中统计。
requests = [{"id": "fin-01", "tenant": "finance", "role": "admin", "acl": 3, "segments": ("system:企业助手v3", "policy:财务报销上限5000", "query:差旅标准")}, {"id": "fin-02", "tenant": "finance", "role": "admin", "acl": 3, "segments": ("system:企业助手v3", "policy:财务报销上限5000", "query:发票期限")}, {"id": "hr-01", "tenant": "hr", "role": "admin", "acl": 7, "segments": ("system:企业助手v3", "policy:仅HR可见薪酬区间", "query:调薪规则")}, {"id": "hr-02", "tenant": "hr", "role": "admin", "acl": 7, "segments": ("system:企业助手v3", "policy:仅HR可见薪酬区间", "query:职级映射")}, {"id": "fin-guest", "tenant": "finance", "role": "guest", "acl": 3, "segments": ("system:企业助手v3", "policy:财务报销上限5000", "query:差旅标准")}, {"id": "hr-v8", "tenant": "hr", "role": "admin", "acl": 8, "segments": ("system:企业助手v3", "policy:仅HR可见薪酬区间", "query:调薪规则")} ]  # 定义六条具有租户和权限版本的脱敏请求。
print("教学实验输入：多租户 Prefix Cache 请求")  # 输出本案例的安全背景。
print("请求       tenant   role   acl  前缀段")  # 输出请求字段表头。
for request in requests:  # 逐条展示命名空间和三段 prompt。
    print(f"{request['id']:<11} {request['tenant']:<8} {request['role']:<6} {request['acl']:>3}  {' | '.join(request['segments'])}")  # 输出当前请求的可读前缀。

教学实验输入：多租户 Prefix Cache 请求
请求       tenant   role   acl  前缀段
fin-01      finance  admin    3  system:企业助手v3 | policy:财务报销上限5000 | query:差旅标准
fin-02      finance  admin    3  system:企业助手v3 | policy:财务报销上限5000 | query:发票期限
hr-01       hr       admin    7  system:企业助手v3 | policy:仅HR可见薪酬区间 | query:调薪规则
hr-02       hr       admin    7  system:企业助手v3 | policy:仅HR可见薪酬区间 | query:职级映射
fin-guest   finance  guest    3  system:企业助手v3 | policy:财务报销上限5000 | query:差旅标准
hr-v8       hr       admin    8  system:企业助手v3 | policy:仅HR可见薪酬区间 | query:调薪规则


## 2. Baseline / 基线：只按文本前缀使用全局缓存

基线把所有租户放进同一个前缀表。它能获得很高命中，却会让 guest 命中 admin 写入的政策 KV，也会让 ACL v8 命中 v7。

In [2]:
global_prefixes = {}  # 创建忽略租户和权限的错误全局缓存。
baseline_rows = []  # 保存逐请求最长前缀命中和来源。
for request in requests:  # 按请求到达顺序查询并写入全局缓存。
    best_length = 0  # 初始化当前请求的最长命中段数。
    best_source = None  # 初始化命中来源以识别越权复用。
    for length in range(1, len(request["segments"]) + 1):  # 从一段到完整 prompt 检查连续前缀。
        prefix = request["segments"][:length]  # 构造当前长度的文本前缀键。
        if prefix in global_prefixes:  # 检查全局表是否已有相同文本。
            best_length = length  # 更新当前最长命中长度。
            best_source = global_prefixes[prefix]  # 记录写入该缓存的安全作用域。
    for length in range(1, len(request["segments"]) + 1):  # 把当前请求的所有连续前缀写入全局表。
        global_prefixes.setdefault(request["segments"][:length], (request["tenant"], request["role"], request["acl"]))  # 保留最早写入来源用于复现泄漏。
    baseline_rows.append({"id": request["id"], "hit": best_length, "source": best_source})  # 保存当前请求命中和来源。
print("Baseline 全局文本键命中")  # 标记当前输出属于不安全缓存。
print("请求       命中段  缓存来源")  # 输出命中表头。
for row in baseline_rows:  # 逐条展示全局缓存来源。
    print(f"{row['id']:<11} {row['hit']:>6}  {row['source']}")  # 输出当前请求最长命中及写入者。

Baseline 全局文本键命中
请求       命中段  缓存来源
fin-01           0  None
fin-02           2  ('finance', 'admin', 3)
hr-01            1  ('finance', 'admin', 3)
hr-02            2  ('hr', 'admin', 7)
fin-guest        3  ('finance', 'admin', 3)
hr-v8            3  ('hr', 'admin', 7)


## 3. 底层实现：命名空间隔离的前缀 Trie

Trie 节点逐段保存 children。根节点按 tenant、role、ACL version 和模型版本隔离，因此相同文本只有在安全作用域一致时才可能复用。

In [3]:
class TrieNode:  # 定义单个前缀节点及其子边。
    def __init__(self):  # 初始化空节点。
        self.children = {}  # 保存 segment 到下一节点的映射。
        self.cached = False  # 标记到当前节点的 KV 是否已经物化。
class IsolatedPrefixCache:  # 实现按安全命名空间隔离的最小 Trie 缓存。
    def __init__(self, model_version):  # 初始化模型版本和命名空间根节点。
        self.model_version = model_version  # 保存 KV 与权重绑定的模型版本。
        self.roots = defaultdict(TrieNode)  # 为每个安全作用域按需创建独立根节点。
    def namespace(self, request):  # 构造不可省略的安全缓存作用域。
        return request["tenant"], request["role"], request["acl"], self.model_version  # 绑定租户、角色、ACL 和模型版本。
    def match_and_insert(self, request):  # 查询最长前缀并写入当前请求路径。
        root = self.roots[self.namespace(request)]  # 只进入当前安全命名空间的 Trie。
        node = root  # 从命名空间根节点开始遍历。
        matched = 0  # 初始化连续命中段数。
        for segment in request["segments"]:  # 按 prompt 顺序逐段查询。
            if segment not in node.children:  # 遇到第一条缺失边时停止命中。
                break  # 保证只复用连续前缀而不是任意子串。
            node = node.children[segment]  # 沿已有边进入下一节点。
            matched += 1 if node.cached else 0  # 只有已物化节点才计入命中。
        node = root  # 回到根节点写入完整请求路径。
        for segment in request["segments"]:  # 逐段创建或复用 Trie 节点。
            node = node.children.setdefault(segment, TrieNode())  # 创建当前缺失前缀节点。
            node.cached = True  # 标记到当前节点的 KV 已经可复用。
        return matched  # 返回安全作用域内的连续命中长度。
    def node_count(self):  # 统计所有命名空间的物化节点数量。
        def count(node):  # 递归统计当前节点及全部后代。
            return 1 + sum(count(child) for child in node.children.values())  # 汇总当前节点和子树大小。
        return sum(count(root) for root in self.roots.values())  # 汇总全部安全命名空间节点。
cache = IsolatedPrefixCache(model_version="model-v17")  # 创建绑定模型版本的隔离缓存。
isolated_rows = []  # 保存每条请求的安全命中结果。
for request in requests:  # 按同一到达顺序执行隔离缓存查询。
    hit = cache.match_and_insert(request)  # 查询并写入当前安全作用域。
    isolated_rows.append({"id": request["id"], "hit": hit, "namespace": cache.namespace(request)})  # 保存命中长度和完整命名空间。
print("隔离 Radix/Trie 命中结果")  # 标记当前输出已经应用安全作用域。
print("请求       命中段  namespace")  # 输出隔离结果表头。
for row in isolated_rows:  # 逐条展示安全命中和命名空间。
    print(f"{row['id']:<11} {row['hit']:>6}  {row['namespace']}")  # 输出当前请求的隔离结果。
print(f"命名空间数={len(cache.roots)}，Trie节点总数={cache.node_count()}")  # 展示隔离带来的结构成本。

隔离 Radix/Trie 命中结果
请求       命中段  namespace
fin-01           0  ('finance', 'admin', 3, 'model-v17')
fin-02           2  ('finance', 'admin', 3, 'model-v17')
hr-01            0  ('hr', 'admin', 7, 'model-v17')
hr-02            2  ('hr', 'admin', 7, 'model-v17')
fin-guest        0  ('finance', 'guest', 3, 'model-v17')
hr-v8            0  ('hr', 'admin', 8, 'model-v17')
命名空间数=4，Trie节点总数=18


## 4. 结果表与结果解读

同租户、同角色、同 ACL 的第二条请求可以复用两段；guest 和 ACL v8 则必须冷启动。缓存命中下降是安全隔离的预期成本，不应当被当作性能回归偷偷取消。

In [4]:
baseline_by_id = {row["id"]: row for row in baseline_rows}  # 建立基线请求索引方便同请求对照。
isolated_by_id = {row["id"]: row for row in isolated_rows}  # 建立隔离缓存请求索引。
print("逐请求命中对照")  # 标记下表使用同一请求比较安全代价。
print("请求       全局命中  隔离命中  结论")  # 输出命中对照表头。
for request in requests:  # 逐请求比较全局和隔离缓存。
    baseline_hit = baseline_by_id[request["id"]]["hit"]  # 读取不安全全局命中长度。
    isolated_hit = isolated_by_id[request["id"]]["hit"]  # 读取安全作用域内命中长度。
    conclusion = "安全复用" if isolated_hit > 0 else "冷启动/隔离"  # 为读者解释当前差异。
    print(f"{request['id']:<11} {baseline_hit:>8} {isolated_hit:>9}  {conclusion}")  # 输出同请求命中对照。
print("解读：fin-02 和 hr-02 保留合法复用；fin-guest 与 hr-v8 牺牲命中以阻止降权或版本越界。")  # 直接说明何种命中应该保留或拒绝。

逐请求命中对照
请求       全局命中  隔离命中  结论
fin-01             0         0  冷启动/隔离
fin-02             2         2  安全复用
hr-01              1         0  冷启动/隔离
hr-02              2         2  安全复用
fin-guest          3         0  冷启动/隔离
hr-v8              3         0  冷启动/隔离
解读：fin-02 和 hr-02 保留合法复用；fin-guest 与 hr-v8 牺牲命中以阻止降权或版本越界。


## 5. 失败案例与修正

fin-guest 的文本与 admin 请求完全相同。全局键会返回 admin 来源的三段 KV；命名空间加入 role 后命中为零，避免低权限请求继承高权限上下文。

In [5]:
guest_baseline = baseline_by_id["fin-guest"]  # 读取 guest 在不安全全局缓存中的命中。
guest_isolated = isolated_by_id["fin-guest"]  # 读取 guest 在隔离 Trie 中的命中。
leaked_from_admin = guest_baseline["source"] is not None and guest_baseline["source"][1] == "admin"  # 判断缓存来源是否为高权限角色。
print(f"错误行为：fin-guest 全局命中={guest_baseline['hit']}，来源={guest_baseline['source']}，越权风险={leaked_from_admin}")  # 展示跨角色复用证据。
print(f"修正行为：namespace={guest_isolated['namespace']}，隔离命中={guest_isolated['hit']}")  # 展示加入 role 和 ACL 后的安全 miss。

错误行为：fin-guest 全局命中=3，来源=('finance', 'admin', 3)，越权风险=True
修正行为：namespace=('finance', 'guest', 3, 'model-v17')，隔离命中=0


## 6. 生产边界

真实 Prefix Cache 存储的是 KV 物理块而非字符串段，还要绑定 tokenizer、rope 配置、adapter、工具 schema 和内容摘要。需要引用计数、TTL/LRU、ACL 撤销广播、哈希校验与审计日志。

In [6]:
production_key_fields = ["tenant", "role", "acl_version", "model_version", "tokenizer_hash", "adapter_id", "tool_schema_hash"]  # 列出生产缓存键不能遗漏的兼容与安全字段。
print("生产缓存键建议：", " + ".join(production_key_fields))  # 输出从教学命名空间扩展到生产的字段。

生产缓存键建议： tenant + role + acl_version + model_version + tokenizer_hash + adapter_id + tool_schema_hash


## 7. 最小回归测试

断言只保护样本数量、合法复用和两个越权 miss。

In [7]:
assert len(requests) >= 5  # 保证案例至少包含五条有安全字段的请求。
assert isolated_by_id["fin-02"]["hit"] == 2  # 保证同作用域重复政策前缀可以合法复用。
assert isolated_by_id["hr-02"]["hit"] == 2  # 保证另一租户内部也能独立复用。
assert guest_isolated["hit"] == 0  # 保证降权 guest 不命中 admin KV。
assert isolated_by_id["hr-v8"]["hit"] == 0  # 保证 ACL 版本变化自动产生冷启动。